In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [6]:
df = pd.read_csv(r"C:\docx\Predictive_Maintenance\data\ai4i+2020+predictive+maintenance+dataset\ai4i2020.csv")

In [7]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [8]:
features = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

target = "Machine failure"

X = df[features].copy()
y = df[target].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nX columns:")
print(X.columns.tolist())

print("\ny distribution:")
print(y.value_counts())

X shape: (10000, 6)
y shape: (10000,)

X columns:
['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

y distribution:
Machine failure
0    9661
1     339
Name: count, dtype: int64


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)

print("\nTraining failure percentages:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting failure percentages:")
print((y_test.value_counts(normalize=True) * 100).round(2))

Training feature shape: (8000, 6)
Testing feature shape: (2000, 6)

Training failure percentages:
Machine failure
0    96.61
1     3.39
Name: proportion, dtype: float64

Testing failure percentages:
Machine failure
0    96.6
1     3.4
Name: proportion, dtype: float64


In [11]:
from sklearn.preprocessing import OneHotEncoder
categorical_features = ["Type"]

numerical_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ]
)

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


## Train Logistic Regression, KNN, and Random Forest

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "KNN": Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", KNeighborsClassifier(
            n_neighbors=5,
            weights="distance"
        ))
    ]),

    "Random Forest": Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name} trained successfully.")

Logistic Regression trained successfully.
KNN trained successfully.
Random Forest trained successfully.


In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

results = []

for name, model in models.items():

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_test, predictions, zero_division=0
        ),
        "ROC AUC": roc_auc_score(y_test, probabilities),
        "PR AUC (Average Precision)": average_precision_score(
            y_test, probabilities
        )
    })

    print(f"\n{'=' * 45}")
    print(name)
    print(classification_report(
        y_test, predictions, zero_division=0
    ))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, predictions))

results_df = pd.DataFrame(results)

display(
    results_df.sort_values(
        "PR AUC (Average Precision)",
        ascending=False
    ).round(4)
)


Logistic Regression
              precision    recall  f1-score   support

           0       0.99      0.82      0.90      1932
           1       0.14      0.82      0.24        68

    accuracy                           0.82      2000
   macro avg       0.57      0.82      0.57      2000
weighted avg       0.96      0.82      0.88      2000

Confusion matrix:
[[1593  339]
 [  12   56]]

KNN
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1932
           1       0.85      0.34      0.48        68

    accuracy                           0.98      2000
   macro avg       0.91      0.67      0.74      2000
weighted avg       0.97      0.98      0.97      2000

Confusion matrix:
[[1928    4]
 [  45   23]]

Random Forest
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      1932
           1       0.79      0.56      0.66        68

    accuracy                           0.98      20

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC,PR AUC (Average Precision)
2,Random Forest,0.9800,0.7917,0.5588,0.6552,0.9614,0.7496
1,KNN,0.9755,0.8519,0.3382,0.4842,0.8302,0.5269
0,Logistic Regression,0.8245,0.1418,0.8235,0.2419,0.9070,0.3818
